In [5]:
# Import necessary Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
import joblib
from pathlib import Path

In [13]:
# Set the main project folder
PROJECT_DIR = Path.cwd().parent

# Define the Task B dataset folders
task_a_path = PROJECT_DIR / "Data" / "TaskA - modified set"

# Read the dataset
df=pd.read_csv(task_a_path / "training" / "english" / "taskA_training_en.tsv",sep="\t")

# Shape of the Dataset

print("Shape of the dataset:", df.shape)

# Display the First 5 Rows of the Dataset
df.head()

Shape of the dataset: (28879, 4)


,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978-e45c-4f28-b859-7f89221b0505,air commodore,flight lieutenant
0,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,command and control officer,flight officer
1,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,air commodore,command and control officer
2,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,pilot officer,squadron leader
3,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,royal airforce officer,command and control officer
4,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,squadron leader,command and control officer


In [3]:
# Rename the Columns of the Dataset
df.columns = ["isco","occupation_uri","jobtitle_1","jobtitle_2"]
df.head()

,isco,occupation_uri,jobtitle_1,jobtitle_2
0,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,air commodore,flight lieutenant
1,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,command and control officer,flight officer
2,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,air commodore,command and control officer
3,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,pilot officer,squadron leader
4,http://data.europa.eu/esco/isco/C0110,http://data.europa.eu/esco/occupation/f2cc5978...,royal airforce officer,command and control officer


In [4]:
# Display the Columns and Information of the Dataset
print(df.columns)
# Display the Information of the Dataset
print(df.info())

Index(['isco', 'occupation_uri', 'jobtitle_1', 'jobtitle_2'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 28880 entries, 0 to 28879
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   isco            28880 non-null  str  
 1   occupation_uri  28880 non-null  str  
 2   jobtitle_1      28880 non-null  str  
 3   jobtitle_2      28880 non-null  str  
dtypes: str(4)
memory usage: 5.3 MB
None


In [5]:
# Display the Unique Values of the Datase
print("Unique ISCO codes:", df['isco'].nunique())
print("Unique occupation URIs:", df['occupation_uri'].nunique())
print("Unique jobtitle_1:", df['jobtitle_1'].nunique())
print("Unique jobtitle_2:", df['jobtitle_2'].nunique())

Unique ISCO codes: 424
Unique occupation URIs: 3011
Unique jobtitle_1: 18863
Unique jobtitle_2: 18754


In [6]:
# Check for Duplicate Rows in the Dataset
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [7]:
# Check for Duplicate Title Pairs in the Dataset
print("Duplicate title pairs:", df[['jobtitle_1', 'jobtitle_2']].duplicated().sum())

Duplicate title pairs: 16


In [8]:
# Find all rows where the same pair of job titles occurs more than once
duplicate_pairs = df[df.duplicated(
    subset=['jobtitle_1', 'jobtitle_2'],
    keep=False
)]

# Sort the duplicate pairs for easier inspection
duplicate_pairs.sort_values(
    ['jobtitle_1', 'jobtitle_2']
)

,isco,occupation_uri,jobtitle_1,jobtitle_2
14289,http://data.europa.eu/esco/isco/C3153,http://data.europa.eu/esco/occupation/523ab5a0...,airline pilot,captain
14299,http://data.europa.eu/esco/isco/C3153,http://data.europa.eu/esco/occupation/522c55ca...,airline pilot,captain
20830,http://data.europa.eu/esco/isco/C6112,http://data.europa.eu/esco/occupation/b865693a...,arboriculturist,arborist
20928,http://data.europa.eu/esco/isco/C6113,http://data.europa.eu/esco/occupation/fa30c1cb...,arboriculturist,arborist
14079,http://data.europa.eu/esco/isco/C3141,http://data.europa.eu/esco/occupation/f3b99acc...,biochemistry technologist,chemistry technician
14087,http://data.europa.eu/esco/isco/C3141,http://data.europa.eu/esco/occupation/95324a4a...,biochemistry technologist,chemistry technician
419,http://data.europa.eu/esco/isco/C1211,http://data.europa.eu/esco/occupation/30f3ea93...,chief financial officer,CFO
431,http://data.europa.eu/esco/isco/C1211,http://data.europa.eu/esco/occupation/b680251e...,chief financial officer,CFO
1150,http://data.europa.eu/esco/isco/C1321,http://data.europa.eu/esco/occupation/3afd6af4...,clothing operations area manager,clothing operations director
1294,http://data.europa.eu/esco/isco/C1321,http://data.europa.eu/esco/occupation/518348e6...,clothing operations area manager,clothing operations director


In [9]:
# Count how many times each occupation URI appears
print(df['occupation_uri'].value_counts().describe())

count    3011.000000
mean        9.591498
std         1.544204
min         1.000000
25%        10.000000
50%        10.000000
75%        10.000000
max        10.000000
Name: count, dtype: float64


In [10]:
# Count how many occupation classes have fewer than 10 records
occupation_counts = df['occupation_uri'].value_counts()

print("Occupations with less than 10 records:",
      (occupation_counts < 10).sum())

Occupations with less than 10 records: 220


In [11]:
# Show how many occupation classes have each low frequency
print(occupation_counts[occupation_counts < 10].value_counts().sort_index())

count
1     31
3     65
6    124
Name: count, dtype: int64


In [12]:
# Find the minimum character length of jobtitle_1
print("Shortest jobtitle_1 length:",
      df['jobtitle_1'].str.len().min())

# Find the minimum character length of jobtitle_2
print("Shortest jobtitle_2 length:",
      df['jobtitle_2'].str.len().min())

Shortest jobtitle_1 length: 2
Shortest jobtitle_2 length: 2


In [13]:
# Display unique jobtitle_1 values containing only 2 characters
print(
    df.loc[
        df['jobtitle_1'].str.len() == 2,
        'jobtitle_1'
    ].unique()
)

# Display unique jobtitle_2 values containing only 2 characters

print(
    df.loc[
        df['jobtitle_2'].str.len() == 2,
        'jobtitle_2'
    ].unique()
)

<ArrowStringArray>
['QS', 'TA', 'DJ', 'PI', 'SD', 'AB', 'OS', 'KP']
Length: 8, dtype: str
<ArrowStringArray>
['TA', 'DJ', 'PA', 'AB', 'OS', 'KP']
Length: 6, dtype: str


In [14]:
# Check how many jobtitle_1 values contain leading or trailing spaces
print(
    "jobtitle_1 with extra spaces:",
    (df['jobtitle_1'] != df['jobtitle_1'].str.strip()).sum()
)

# Check how many jobtitle_2 values contain leading or trailing spaces
print(
    "jobtitle_2 with extra spaces:",
    (df['jobtitle_2'] != df['jobtitle_2'].str.strip()).sum()
)

jobtitle_1 with extra spaces: 63
jobtitle_2 with extra spaces: 62


In [15]:
# Create a separate copy so that the original dataset remains unchanged
df_preprocessed = df.copy()

In [16]:
# Convert Job Titles into Lowercase
df_preprocessed['jobtitle_1'] = df_preprocessed['jobtitle_1'].str.lower()
df_preprocessed['jobtitle_2'] = df_preprocessed['jobtitle_2'].str.lower()

In [17]:
# Remove leading and trailing spaces from Job Titles
df_preprocessed['jobtitle_1'] = df_preprocessed['jobtitle_1'].str.strip()
df_preprocessed['jobtitle_2'] = df_preprocessed['jobtitle_2'].str.strip()

In [18]:
# Display the first 10 rows of the preprocessed job titles
df_preprocessed[['jobtitle_1', 'jobtitle_2']].head(10)

,jobtitle_1,jobtitle_2
0,air commodore,flight lieutenant
1,command and control officer,flight officer
2,air commodore,command and control officer
3,pilot officer,squadron leader
4,royal airforce officer,command and control officer
5,squadron leader,command and control officer
6,military group captain,flight officer
7,squadron leader,flight officer
8,flight lieutenant,royal airforce officer
9,air commodore,squadron leader


In [19]:
# Check whether jobtitle_1 or jobtitle_2 contains missing values
df_preprocessed[['jobtitle_1', 'jobtitle_2']].isna().sum()

jobtitle_1    0
jobtitle_2    0
dtype: int64

In [20]:
# Count how many samples are available for each occupation URI
class_counts = df_preprocessed['occupation_uri'].value_counts()

In [21]:
# Find occupation classes that have only one sample
rare_classes = class_counts[class_counts == 1].index

# Create a dataframe containing the single-sample classes
rare_df = df_preprocessed[
    df_preprocessed['occupation_uri'].isin(rare_classes)
]

# Create another dataframe containing all remaining classes
regular_df = df_preprocessed[
    ~df_preprocessed['occupation_uri'].isin(rare_classes)
]

In [22]:
# Display the number of rows and columns in the rare and regular datasets
print("Rare dataframe shape:", rare_df.shape)
print("Regular dataframe shape:", regular_df.shape)

Rare dataframe shape: (31, 4)
Regular dataframe shape: (28849, 4)


In [23]:
# Split the regular classes into training and testing data
train_regular, test_df = train_test_split(
    regular_df,
    test_size=0.2,
    random_state=42,
    stratify=regular_df['occupation_uri']
)

In [24]:
# Add all single-sample occupation classes to the training set
train_df = pd.concat(
    [train_regular, rare_df],
    ignore_index=True
)

In [25]:
# Display the final training and testing dataset size
print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

Training set shape: (23110, 4)
Testing set shape: (5770, 4)


In [26]:
# Find occupation classes that appear in the test set but do not appear in the training set
test_only_classes = set(test_df['occupation_uri']) - set(train_df['occupation_uri'])

# Display the number of test-only classes
print("Classes present only in test set:",len(test_only_classes))

Classes present only in test set: 0


In [27]:
# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer()

# Combine both job-title columns from the training data so that both title columns use the same vocabulary
train_titles = pd.concat([train_df['jobtitle_1'],train_df['jobtitle_2']])

# Learn the vocabulary and IDF values from training data
tfidf.fit(train_titles)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [28]:
# Convert jobtitle_1 and jobtitle_2from the training set into TF-IDF features
X_train_title1 = tfidf.transform(train_df['jobtitle_1'])
X_train_title2 = tfidf.transform(train_df['jobtitle_2'])

In [29]:
# Convert jobtitle_1 and jobtitle_2from the test set using the vocabulary already learned from the training data
X_test_title1 = tfidf.transform(test_df['jobtitle_1'])
X_test_title2 = tfidf.transform(test_df['jobtitle_2'])

In [30]:
# Display the shapes of TF-IDF features for jobtitle_1 and jobtitle_2 in training and testing data
print("X_train_title1 shape:", X_train_title1.shape)
print("X_train_title2 shape:", X_train_title2.shape)
print("X_test_title1 shape:", X_test_title1.shape)
print("X_test_title2 shape:", X_test_title2.shape)

X_train_title1 shape: (23110, 5424)
X_train_title2 shape: (23110, 5424)
X_test_title1 shape: (5770, 5424)
X_test_title2 shape: (5770, 5424)


In [31]:
# Combine TF-IDF features of jobtitle_1 and jobtitle_2 for the training and testing data
X_train = hstack([X_train_title1,X_train_title2])
X_test = hstack([X_test_title1,X_test_title2])

In [32]:
# Display the final training and testing feature matrix shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (23110, 10848)
X_test shape: (5770, 10848)


In [33]:
# Select occupation_uri as the target variable for training and testing
y_train = train_df['occupation_uri']
y_test = test_df['occupation_uri']

In [34]:
# Display the number of target values in the training set and testing set
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

y_train shape: (23110,)
y_test shape: (5770,)


In [35]:
# Create the label encoder
label_encoder = LabelEncoder()

# Learn the occupation classes from the training data and convert them into numerical labels
y_train_encoded = label_encoder.fit_transform(y_train)

# Convert test occupation URIs using the same encoder
y_test_encoded = label_encoder.transform(y_test)

In [36]:
# Display the total number of occupation classes
print("Number of classes:", len(label_encoder.classes_))

# Display the shape of encoded training and testing targets
print("y_train_encoded shape:", y_train_encoded.shape)
print("y_test_encoded shape:", y_test_encoded.shape)

Number of classes: 3011
y_train_encoded shape: (23110,)
y_test_encoded shape: (5770,)


In [37]:
# Create the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000,random_state=42)

In [38]:
# Train the Logistic Regression model using the Training Data
logistic_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [39]:
# Make predictions using the trained Logistic Regression Model
y_pred_logistic = logistic_model.predict(X_test)

# Display the first 10 predicted occupation classes
print("First 10 predictions:", y_pred_logistic[:10])

First 10 predictions: ['http://data.europa.eu/esco/occupation/1bb2aa65-1524-4961-8dba-ad25f34f195f'
 'http://data.europa.eu/esco/occupation/4fef1907-cf82-413b-841a-02ef57e299a6'
 'http://data.europa.eu/esco/occupation/65fb0b53-b8c4-4839-b099-c80b6de5a158'
 'http://data.europa.eu/esco/occupation/a273ec4c-96fd-48e5-9343-695a6bd08512'
 'http://data.europa.eu/esco/occupation/f89b12a1-689f-4a4f-8695-abb620810f1e'
 'http://data.europa.eu/esco/occupation/bcc199fe-f7df-42a5-bd45-e33ae6160f26'
 'http://data.europa.eu/esco/occupation/cb3b2662-774a-4e6f-841e-f120244d7031'
 'http://data.europa.eu/esco/occupation/b06a1dde-e5a1-4219-9491-2bedd0282e4b'
 'http://data.europa.eu/esco/occupation/1d70e660-1114-41c6-926a-053541460d88'
 'http://data.europa.eu/esco/occupation/2c83f279-a1d1-4c9f-a6aa-5e5ee1850e46']


In [40]:
# Calculate the accuracy of the Logistic Regression model
logistic_accuracy = accuracy_score(y_test, y_pred_logistic)

# Display the accuracy
print("Logistic Regression Accuracy:", logistic_accuracy)

Logistic Regression Accuracy: 0.8162911611785095


In [41]:
# Calculate weighted precision
logistic_precision = precision_score(
    y_test,
    y_pred_logistic,
    average='weighted',
    zero_division=0
)

# Calculate weighted recall
logistic_recall = recall_score(
    y_test,
    y_pred_logistic,
    average='weighted',
    zero_division=0
)

# Calculate weighted F1-score
logistic_f1 = f1_score(
    y_test,
    y_pred_logistic,
    average='weighted',
    zero_division=0
)

# Display the evaluation results
print("Logistic Regression Precision:", logistic_precision)
print("Logistic Regression Recall:", logistic_recall)
print("Logistic Regression F1-score:", logistic_f1)

Logistic Regression Precision: 0.8124065362713544
Logistic Regression Recall: 0.8162911611785095
Logistic Regression F1-score: 0.7916544249126573


In [42]:
# Create the Linear SVM model
svm_model = LinearSVC(C=1.0,random_state=42,max_iter=2000)

In [43]:
# Train the Linear SVM model using the training TF-IDF features
svm_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_iter max_iter: int, default=1000The maximum number of iterations to be run.",2000
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are su

In [44]:
# Make predictions on the unseen test data using the trained Linear SVM model
y_pred_svm = svm_model.predict(X_test)

# Display the first 10 predictions to verify that the model is working
print("First 10 SVM predictions:", y_pred_svm[:10])

First 10 SVM predictions: ['http://data.europa.eu/esco/occupation/1bb2aa65-1524-4961-8dba-ad25f34f195f'
 'http://data.europa.eu/esco/occupation/31cc4199-703c-4f8a-9b46-a2070a6d35df'
 'http://data.europa.eu/esco/occupation/65fb0b53-b8c4-4839-b099-c80b6de5a158'
 'http://data.europa.eu/esco/occupation/a273ec4c-96fd-48e5-9343-695a6bd08512'
 'http://data.europa.eu/esco/occupation/f89b12a1-689f-4a4f-8695-abb620810f1e'
 'http://data.europa.eu/esco/occupation/bcc199fe-f7df-42a5-bd45-e33ae6160f26'
 'http://data.europa.eu/esco/occupation/cb3b2662-774a-4e6f-841e-f120244d7031'
 'http://data.europa.eu/esco/occupation/b06a1dde-e5a1-4219-9491-2bedd0282e4b'
 'http://data.europa.eu/esco/occupation/1d70e660-1114-41c6-926a-053541460d88'
 'http://data.europa.eu/esco/occupation/2c83f279-a1d1-4c9f-a6aa-5e5ee1850e46']


In [45]:
# Calculate the accuracy of the Linear SVM model
svm_accuracy = accuracy_score(y_test, y_pred_svm)

# Display the accuracy
print("Linear SVM Accuracy:", svm_accuracy)

Linear SVM Accuracy: 0.891161178509532


In [46]:
# Calculate Precision, Recall, and F1-score for the Linear SVM model
svm_precision = precision_score(
    y_test,
    y_pred_svm,
    average='weighted',
    zero_division=0
)

svm_recall = recall_score(
    y_test,
    y_pred_svm,
    average='weighted',
    zero_division=0
)

svm_f1 = f1_score(
    y_test,
    y_pred_svm,
    average='weighted',
    zero_division=0
)

# Display the evaluation results
print("Linear SVM Precision:", svm_precision)
print("Linear SVM Recall:", svm_recall)
print("Linear SVM F1-score:", svm_f1)

Linear SVM Precision: 0.9004968226458695
Linear SVM Recall: 0.891161178509532
Linear SVM F1-score: 0.880312783692333


In [47]:
# Create the Multinomial Naive Bayes model
nb_model = MultinomialNB(alpha=1.0)

In [48]:
# Train the Multinomial Naive Bayes model using the training TF-IDF features
nb_model.fit(X_train,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](3011,)","[8.,8.,8.,...,8.,5.,8.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](3011,)","[-7.97,-7.97,-7.97,...,-7.97,-8.44,-7.97]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U74](3011,)","['http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34', 'http://data.europa.eu/esco/occupation/000e93a3-d956-4e45-aacb-f12c83fedf84', 'http://data.europa.eu/esco/occupation/0019b951-c699-4191-8208-9822882d150c', ..., 'http://data.europa.eu/esco/occupation/ffa4dd5d-702b-4828-ad85-17ec50db6656', 'http://data.europa.eu/esco/occupation/ffade2f4-2528-4dfd-b36a-6e4092be08ed', 'http://data.europa.eu/esco/occupation/ffc292f1-89e2-4406-8eff-39096c94e731']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](3011, 10848)","[[0.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,0.,0.], ..., [0.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,0.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](3011, 10848)","[[-9.29,-9.29,-9.29,...,-9.29,-9.29,-9.29], [-9.29,-9.29,-9.29,...,-9.29,-9.29,-9.29], [-9.29,-9.29,-9.29,...,-9.29,-9.29,-9.29], ..., [-9.29,-9.29,-9.29,...,-9.29,-9.29,-9.29], [-9.29,-9.29,-9.29,...,-9.29,-9.29,-9.29], [-9.29,-9.29,-9.29,...,-9.29,-9.29,-9.29]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10848


In [49]:
# Make predictions on the unseen test data using the trained Naive Bayes model
y_pred_nb = nb_model.predict(X_test)

# Display the first 10 predictions to verify that the model is working
print("First 10 Naive Bayes predictions:", y_pred_nb[:10])

First 10 Naive Bayes predictions: ['http://data.europa.eu/esco/occupation/1bb2aa65-1524-4961-8dba-ad25f34f195f'
 'http://data.europa.eu/esco/occupation/4fef1907-cf82-413b-841a-02ef57e299a6'
 'http://data.europa.eu/esco/occupation/65fb0b53-b8c4-4839-b099-c80b6de5a158'
 'http://data.europa.eu/esco/occupation/a273ec4c-96fd-48e5-9343-695a6bd08512'
 'http://data.europa.eu/esco/occupation/f89b12a1-689f-4a4f-8695-abb620810f1e'
 'http://data.europa.eu/esco/occupation/bcc199fe-f7df-42a5-bd45-e33ae6160f26'
 'http://data.europa.eu/esco/occupation/cb3b2662-774a-4e6f-841e-f120244d7031'
 'http://data.europa.eu/esco/occupation/b06a1dde-e5a1-4219-9491-2bedd0282e4b'
 'http://data.europa.eu/esco/occupation/1d70e660-1114-41c6-926a-053541460d88'
 'http://data.europa.eu/esco/occupation/2c83f279-a1d1-4c9f-a6aa-5e5ee1850e46']


In [50]:
# Calculate the accuracy of the Multinomial Naive Bayes model
nb_accuracy = accuracy_score(y_test, y_pred_nb)

# Display the accuracy

print("Multinomial Naive Bayes Accuracy:", nb_accuracy)

Multinomial Naive Bayes Accuracy: 0.8672443674176776


In [51]:
# Calculate Precision, Recall, and F1-score for the Naive Bayes model
nb_precision = precision_score(
    y_test,
    y_pred_nb,
    average='weighted',
    zero_division=0
)

nb_recall = recall_score(
    y_test,
    y_pred_nb,
    average='weighted',
    zero_division=0
)

nb_f1 = f1_score(
    y_test,
    y_pred_nb,
    average='weighted',
    zero_division=0
)

# Display all three evaluation metrics

print("Multinomial Naive Bayes Precision:", nb_precision)
print("Multinomial Naive Bayes Recall:", nb_recall)
print("Multinomial Naive Bayes F1-score:", nb_f1)

Multinomial Naive Bayes Precision: 0.8697177519187917
Multinomial Naive Bayes Recall: 0.8672443674176776
Multinomial Naive Bayes F1-score: 0.849258617369536


In [52]:
# Create a comparison table for all three classification models
model_results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Linear SVM',
        'Multinomial Naive Bayes'
    ],
    'Accuracy': [
        logistic_accuracy,
        svm_accuracy,
        nb_accuracy
    ],
    'Precision': [
        logistic_precision,
        svm_precision,
        nb_precision
    ],
    'Recall': [
        logistic_recall,
        svm_recall,
        nb_recall
    ],
    'F1-score': [
        logistic_f1,
        svm_f1,
        nb_f1
    ]
})

# Display the comparison table
model_results

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.816291,0.812407,0.816291,0.791654
1,Linear SVM,0.891161,0.900497,0.891161,0.880313
2,Multinomial Naive Bayes,0.867244,0.869718,0.867244,0.849259


In [54]:
# We are testing a small range to find a better setting without making the search unnecessarily large
param_grid = {
    'C': [0.1, 0.5, 1.0, 2.0, 5.0]
    }

# Display the hyperparameter values that will be tested
print("C values to test:", param_grid['C'])

C values to test: [0.1, 0.5, 1.0, 2.0, 5.0]


In [55]:
# Create the Linear SVM model that GridSearchCV will tune
svm_grid = GridSearchCV(
    estimator=LinearSVC(
        random_state=42,
        max_iter=2000
    ),
    param_grid=param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

In [56]:
# Run GridSearchCV on the training data
svm_grid.fit(X_train, y_train)

C:\Users\jagad\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LinearSVC(max...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_weighted'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with 

In [57]:
# Display the best C value found by GridSearchCV
print("Best C:", svm_grid.best_params_['C'])

# Display the best weighted F1-score obtained during cross-validation
print("Best cross-validation F1-score:", svm_grid.best_score_)

Best C: 5.0
Best cross-validation F1-score: 0.8177533134122198


In [58]:
# Use the best Linear SVM model found by GridSearchCV
y_pred_svm_tuned = svm_grid.best_estimator_.predict(X_test)

# Display the first 10 predictions to confirm that the tuned model is working
print("First 10 tuned SVM predictions:", y_pred_svm_tuned[:10])

First 10 tuned SVM predictions: ['http://data.europa.eu/esco/occupation/1bb2aa65-1524-4961-8dba-ad25f34f195f'
 'http://data.europa.eu/esco/occupation/f3b99acc-b212-45cf-bb4f-7b83cc6f0d71'
 'http://data.europa.eu/esco/occupation/65fb0b53-b8c4-4839-b099-c80b6de5a158'
 'http://data.europa.eu/esco/occupation/a273ec4c-96fd-48e5-9343-695a6bd08512'
 'http://data.europa.eu/esco/occupation/f89b12a1-689f-4a4f-8695-abb620810f1e'
 'http://data.europa.eu/esco/occupation/bcc199fe-f7df-42a5-bd45-e33ae6160f26'
 'http://data.europa.eu/esco/occupation/cb3b2662-774a-4e6f-841e-f120244d7031'
 'http://data.europa.eu/esco/occupation/b06a1dde-e5a1-4219-9491-2bedd0282e4b'
 'http://data.europa.eu/esco/occupation/1d70e660-1114-41c6-926a-053541460d88'
 'http://data.europa.eu/esco/occupation/2c83f279-a1d1-4c9f-a6aa-5e5ee1850e46']


In [59]:
# Calculate Accuracy, Precision, Recall, and F1-score for the tuned Linear SVM
tuned_svm_accuracy = accuracy_score(y_test, y_pred_svm_tuned)

tuned_svm_precision = precision_score(
    y_test,
    y_pred_svm_tuned,
    average='weighted',
    zero_division=0
)

tuned_svm_recall = recall_score(
    y_test,
    y_pred_svm_tuned,
    average='weighted',
    zero_division=0
)

tuned_svm_f1 = f1_score(
    y_test,
    y_pred_svm_tuned,
    average='weighted',
    zero_division=0
)

# Display the test-set performance of the tuned model
print("Tuned Linear SVM Accuracy:", tuned_svm_accuracy)
print("Tuned Linear SVM Precision:", tuned_svm_precision)
print("Tuned Linear SVM Recall:", tuned_svm_recall)
print("Tuned Linear SVM F1-score:", tuned_svm_f1)

Tuned Linear SVM Accuracy: 0.8935875216637782
Tuned Linear SVM Precision: 0.9025534373194686
Tuned Linear SVM Recall: 0.8935875216637782
Tuned Linear SVM F1-score: 0.8829454485433689


In [60]:
# Add the tuned Linear SVM results to the model comparison table
tuned_svm_result = pd.DataFrame({
    'Model': ['Tuned Linear SVM'],
    'Accuracy': [tuned_svm_accuracy],
    'Precision': [tuned_svm_precision],
    'Recall': [tuned_svm_recall],
    'F1-score': [tuned_svm_f1]
})

# Combine the original model results with the tuned SVM results
model_results = pd.concat(
    [model_results, tuned_svm_result],
    ignore_index=True
)

# Display the complete comparison table
model_results

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.816291,0.812407,0.816291,0.791654
1,Linear SVM,0.891161,0.900497,0.891161,0.880313
2,Multinomial Naive Bayes,0.867244,0.869718,0.867244,0.849259
3,Tuned Linear SVM,0.893588,0.902553,0.893588,0.882945


In [61]:
# Sort all models from highest to lowest F1-score
model_results = model_results.sort_values(
    by='F1-score',
    ascending=False
).reset_index(drop=True)

# Display the final sorted comparison table
model_results

,Model,Accuracy,Precision,Recall,F1-score
0,Tuned Linear SVM,0.893588,0.902553,0.893588,0.882945
1,Linear SVM,0.891161,0.900497,0.891161,0.880313
2,Multinomial Naive Bayes,0.867244,0.869718,0.867244,0.849259
3,Logistic Regression,0.816291,0.812407,0.816291,0.791654


In [63]:
# Save the best tuned Linear SVM model to a file
joblib.dump(svm_grid.best_estimator_,'tuned_linear_svm_model.pkl')

# Confirm that the model has been saved
print("Tuned Linear SVM model saved successfully.")

Tuned Linear SVM model saved successfully.


In [64]:
# Save the TF-IDF vectorizer
joblib.dump(tfidf,'tfidf_vectorizer.pkl')

# Confirm that the vectorizer has been saved
print("TF-IDF vectorizer saved successfully.")

TF-IDF vectorizer saved successfully.
